<br>
<a href="https://github.com/aperture-systems-lab">
    <img src="assets/banner_semillero.png" width="955" style="margin: 0px 0px 12px;"/>
</a>
<h1 style="line-height: 1.4;"><font color="#29c4d9"><b>De Prompts a Agentes:</b></font> Inteligencia Artificial Agéntica con Python</h1>
<h2><b>Notebook 1: </b>Fundamentos</h2>
<br>

Conectar un LLM y ver qué cambia cuando movemos sus parámetros.

In [14]:
import warnings

from dotenv import load_dotenv

load_dotenv()
warnings.filterwarnings("ignore", message=".*fixed sampling defaults")

----

<br>

## **Parte 1:** Conectar el modelo

`init_chat_model` conecta cualquier proveedor, aquí Gemini.

In [15]:
from langchain.chat_models import init_chat_model

modelo = init_chat_model("google_genai:gemini-3.6-flash")

In [16]:
respuesta = modelo.invoke("¿Cuál es el barrio más tradicional de Medellín? En una frase.")

respuesta

AIMessage(content=[{'type': 'text', 'text': 'El barrio más tradicional e histórico de Medellín es **El Prado**, reconocido por su arquitectura patrimonial republicana y por ser el primer barrio planeado de la ciudad.', 'extras': {'signature': 'ErUQCrIQAWkUfROoAQHForEVzgjpIFqbo20mu5SVnqWXKe80ZdOUbwhe5Zel5sVXz4yUKo81KljmWYqVvaIrL1Hmf3gxcn4MtPLg/WXe4pNwJm0m3KCwFgt3IfxVWU98BD032fCikwwjYOOvP9SreyTEBixc4JyWsFHofAK92bp90d9ntKnJv82OvpRqF6QRhcSKieyTfkhwuw0HlZsrJHlKUsCXxTVQTH9HZNPpNQ4fTRpPqc5MjO/fdu/b1LN2LLeFk9/o8ezhd8s4JTlzodqVdHNzvEvTSGJL/DLFGQixABIerI6IFcMD0EaZwOYzk7B2S5kJTZB6x0M2Z5JjdGgTQPw5NaS5fQ9vJHtTmlC0LGrj5L9slTdaIf6FOB55RLhiDkh0apcYYZWcIz5NxcbklxXmb+ci8Avjvq99jJuuVgSwKnSP5s6niMxOXm6rINodXaRga1RfMRgO79w5Wi5znv7GtFcTtcXKSZpCxn40DRpKKKQm5Jv9sqSgbOG/FHXvKhzY3oVskt1MbvD+StCmyh2apG13ex/x4JUg7/Kkp9w1Dmpa7PwpQEdvE7eSR0PA4xSC1bwlO7LbgpLeqJ/QfS+JbxhRywp0t3bli96P2OzRUZ+lqRuqlg/rY2HyO/GE7eotl/2rvzy/x70XD8IJKvMkpqIfxWQ1kPBCxT26/y9TJ9046G4CLSROOfyjiBk//8p+HNX2nDFAveZHZUdLuyARIwrTBMT+LD4UUj91j+rFfEjT

In [17]:
print(respuesta.text)

El barrio más tradicional e histórico de Medellín es **El Prado**, reconocido por su arquitectura patrimonial republicana y por ser el primer barrio planeado de la ciudad.


In [18]:
from pprint import pprint

pprint(respuesta.response_metadata)

{'finish_reason': 'STOP',
 'model_name': 'gemini-3.6-flash',
 'model_provider': 'google_genai',
 'safety_ratings': []}


In [19]:
respuesta.usage_metadata

{'input_tokens': 15,
 'output_tokens': 467,
 'total_tokens': 482,
 'input_token_details': {'cache_read': 0},
 'output_token_details': {'reasoning': 434}}

Lo mismo con la clase propia de Gemini.

In [20]:
from langchain_google_genai import ChatGoogleGenerativeAI

modelo = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

print(modelo.invoke("¿Qué significa estrato 5 en Colombia? En una frase.").text)

El **estrato 5** en Colombia representa a la población de ingresos altos o medio-altos que habita en zonas acomodadas y paga una contribución adicional en sus servicios públicos para subsidiar a los estratos de menores recursos.


----

<br>

## **Parte 2:** Mensajes

El modelo recibe una lista de mensajes, cada uno con su rol.

<img src="assets/diagramas/mensajes.png" width="760"/>

In [21]:
from langchain.messages import AIMessage, HumanMessage

mensajes = [
    HumanMessage("Busco apartamento cerca del metro en Medellín."),
    AIMessage("¡Claro! ¿En qué zona de la ciudad?"),
    HumanMessage("Laureles."),
]

print(modelo.invoke(mensajes).text)

¡Excelente elección! Laureles es una de las mejores zonas de Medellín por su seguridad, oferta gastronómica, parques y plano plano (muy caminable).

Un detalle importante sobre la movilidad en esta zona: el **Metro (Línea B)** no pasa por el corazón residencial de Laureles, sino por sus bordes. Las estaciones más cercanas son:

1. **Estadio:** La más cercana a la zona del Primer y Segundo Parque de Laureles y la Avenida Nutibara.
2. **Suramericana:** Muy conveniente si estás cerca de la Universidad UPB o Carlos E. Restrepo.
3. **Floresta:** Cerca de sectores como La Castellana o Laureles occidental.
*(Nota: También cuentas con el sistema **Metroplús**, que cruza por la Avenida 33 y la Calle 30).*

Para ayudarte a filtrar opciones o darte recomendaciones concretas, cuéntame:

1. **¿Buscas arriendo o compra?** (Y si es arriendo, ¿por meses o contrato a un año?).
2. **¿Cuál es tu presupuesto aproximado?**
3. **¿Lo necesitas amoblado o sin amoblar?**
4. **¿Cuántas habitaciones/baños necesi

----

<br>

## **Parte 3:** Temperatura

Con temperatura 0 el modelo repite y con 2 se arriesga.

<img src="assets/diagramas/temperatura.png" width="760"/>

In [22]:
prompt = "Inventa un nombre para un edificio nuevo en Laureles. Responde solo el nombre."

for temperatura in [0.0, 1.0, 2.0]:
    modelo = init_chat_model("google_genai:gemini-3.1-flash-lite", temperature=temperatura, thinking_level="minimal")
    nombres = {modelo.invoke(prompt).text for _ in range(10)}
    print(temperatura, len(nombres), "distintos:", nombres)

0.0 1 distintos: {'Altos de Laureles'}
1.0 5 distintos: {'Aura Laureles', 'Altos de Laureles', 'Aurum Laureles', 'Reserva Laureles', 'Boreal Laureles'}
2.0 10 distintos: {'Laureano', 'Bambú Laureles', 'Bruma Laureles', 'Altos del Laurel', 'Altos de Laureles', 'Índigo Laureles', 'Laureares', 'Vivero Laureles', 'Guayacanes del Laurel', 'Boreal Laureles'}


----

<br>

## **Parte 4:** Límite de tokens y razonamiento

`max_tokens` corta la respuesta, y el razonamiento también gasta de ahí.

<img src="assets/diagramas/limite_tokens.png" width="760"/>

In [23]:
modelo = init_chat_model("google_genai:gemini-3.6-flash", max_tokens=60, thinking_level="minimal")

respuesta = modelo.invoke("Describe el barrio Laureles en un párrafo.")
print(respuesta.text)
print(respuesta.response_metadata["finish_reason"])

El barrio Laureles es uno de los sectores más emblemáticos, tradicionales y acogedores de Medellín, famoso por su diseño urbano de calles arboladas, avenidas circulares y un ambiente predominantemente residencial pero lleno de vida. Reconocido por sus imponentes árboles de
MAX_TOKENS


`thinking_level` decide cuánto piensa el modelo:

- `minimal`
- `low`
- `medium`
- `high`

In [24]:
import time

pregunta = "Un apartamento cuesta 380 millones. Con 30 % de cuota inicial, crédito a 20 años y tasa del 12 % efectivo anual, ¿cuál es la cuota mensual?"

for nivel in ["low", "high"]:
    modelo = init_chat_model("google_genai:gemini-3.6-flash", thinking_level=nivel)
    inicio = time.perf_counter()
    respuesta = modelo.invoke(pregunta)
    razonamiento = respuesta.usage_metadata.get("output_token_details", {}).get("reasoning", 0)
    print(f"{nivel:5} {time.perf_counter() - inicio:5.1f} s   razonamiento: {razonamiento:5} tokens")

low     8.4 s   razonamiento:   889 tokens
high   13.5 s   razonamiento:  1963 tokens


----

<br>

## **Parte 5:** Salida estructurada

En vez de texto, un objeto de Python con los campos que pedimos.

<img src="assets/diagramas/salida_estructurada.png" width="760"/>

In [25]:
from pydantic import BaseModel, Field


class Ficha(BaseModel):
    barrio: str
    alcobas: int
    precio_millones: int = Field(description="Precio de venta en millones de pesos")
    admite_mascotas: bool


anuncio = "Vendo apto en Laureles, 3 habitaciones, 2 baños, 92 m2, 380 milloncitos. ¡Los gatos son bienvenidos!"

ficha = modelo.with_structured_output(Ficha).invoke(anuncio)
ficha

Ficha(barrio='Laureles', alcobas=3, precio_millones=380, admite_mascotas=True)

In [26]:
print(f"{ficha.barrio}: {ficha.alcobas} alcobas por {ficha.precio_millones} millones")

Laureles: 3 alcobas por 380 millones


-----

<br>

### **Siguiente notebook:**

[`02_anatomia_agente.ipynb`](02_anatomia_agente.ipynb)

### <font color="#29c4d9">**Notebook 1 listo.**</font>

<br>

---
<div style="margin-top: 50px;"><center><a href="https://github.com/aperture-systems-lab"><img src="assets/banner_logo.png" width="955"/></a></center></div>